# Activation Fingerprinting

Characterize model behavior through compact activation signatures:
layer fingerprints, head outputs, MLP patterns, attention patterns, and input sensitivity.

## Why This Matters

Activation fingerprinting characterizes the patterns and statistics of internal model activations. Activation structure reveals how the model processes information — which components are active, how sparse the computation is, and what geometric patterns emerge.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.activation_fingerprinting import (
    layer_activation_fingerprint, head_output_fingerprint,
    mlp_activation_fingerprint, attention_pattern_fingerprint,
    input_sensitivity_fingerprint,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Layer Activation Fingerprint

Compact summary of each layer: norm, direction similarity, pairwise similarity.

In [ ]:
result = layer_activation_fingerprint(model, tokens)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: norm={p['mean_norm']:.4f} (±{p['std_norm']:.4f}), "
          f"cos_1st_last={p['cos_first_last']:+.4f}, "
          f"mean_sim={p['mean_pairwise_similarity']:.4f}")

## Head Output Fingerprint

Characterize each head's output: norm, max norm, direction consistency.

In [ ]:
for layer in range(4):
    result = head_output_fingerprint(model, tokens, layer=layer)
    print(f"Layer {layer}:")
    for h in result['per_head']:
        print(f"  H{h['head']}: norm={h['mean_norm']:.4f}, "
              f"max={h['max_norm']:.4f}, "
              f"consistency={h['direction_consistency']:.4f}")
    print()

## MLP Activation Fingerprint

Sparsity, magnitude, and active neuron count per layer.

In [ ]:
result = mlp_activation_fingerprint(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(p['sparsity'] * 20)
    print(f"Layer {p['layer']}: active={p['mean_active_neurons']:.1f}, "
          f"sparsity={p['sparsity']:.2%}, "
          f"mag={p['mean_magnitude']:.4f} {bar}")

## Attention Pattern Fingerprint

Entropy, self/prev/BOS attention for every head.

In [ ]:
result = attention_pattern_fingerprint(model, tokens)
for layer_info in result['per_layer']:
    print(f"Layer {layer_info['layer']}:")
    for h in layer_info['per_head']:
        print(f"  H{h['head']}: entropy={h['mean_entropy']:.3f}, "
              f"self={h['self_attention']:.3f}, "
              f"prev={h['prev_token_attention']:.3f}, "
              f"bos={h['bos_attention']:.3f}")
    print()

## Input Sensitivity Fingerprint

How much does each layer change the residual stream?

In [ ]:
result = input_sensitivity_fingerprint(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(min(p['relative_change'], 1) * 20)
    print(f"Layer {p['layer']}: delta={p['mean_delta_norm']:.4f}, "
          f"rel_change={p['relative_change']:.4f}, "
          f"dir_preserved={p['direction_preservation']:.4f} {bar}")